In [1]:
import boto3
import pandas as pd
from botocore.exceptions import NoCredentialsError, PartialCredentialsError

# Configuración de la región
REGION = 'us-west-2' # Cambia según tu preferencia

def get_ec2_client():
    """Inicializa el cliente de EC2."""
    try:
        return boto3.client('ec2', region_name=REGION)
    except Exception as e:
        print(f"Error al conectar con AWS: {e}")
        return None

In [2]:
def fetch_ec2_instances():
    """
    Obtiene la lista de instancias EC2 y extrae:
    ID, Estado, Tipo e IP Pública.
    """
    ec2 = get_ec2_client()
    if not ec2:
        return []

    instances_data = []
    
    try:
        # Paginación automática con describe_instances
        response = ec2.describe_instances()
        
        for reservation in response['Reservations']:
            for instance in reservation['Instances']:
                instance_info = {
                    "Instance ID": instance.get('InstanceId'),
                    "Estado": instance.get('State', {}).get('Name'),
                    "Tipo": instance.get('InstanceType'),
                    "IP Pública": instance.get('PublicIpAddress', 'N/A') # Algunas instancias no tienen IP pública
                }
                instances_data.append(instance_info)
                
        return instances_data

    except (NoCredentialsError, PartialCredentialsError):
        print("Error: No se encontraron credenciales de AWS configuradas.")
    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")
    return []

In [4]:
def main():
    print(f"--- Monitoreo de Instancias EC2 ({REGION}) ---")
    data = fetch_ec2_instances()
    
    if data:
        df = pd.DataFrame(data)
        # Estilización básica para el Notebook
        display(df.style.set_properties(**{'text-align': 'left'})
                .set_table_styles([dict(selector='th', props=[('text-align', 'center')])]))
    else:
        print("No se encontraron instancias o hubo un error en la conexión.")

if __name__ == "__main__":
    main()

--- Monitoreo de Instancias EC2 (us-west-2) ---


,Instance ID,Estado,Tipo,IP Pública
0,i-0d409e71255e56cfb,running,t3.nano,52.37.51.19
